In [41]:
import os
import pandas as pd

In [42]:
exp_root = "/media/yesindeed/DATADRIVE1/mount/remote_cse/experiments/med_vlm_benchmark/merged"

datasets = {
    "SLAKE": "/research/d5/gds/yzhong22/datasets/SLAKE/imgs",
    "PathVQA": "None",
    "VQA-RAD": "None",
    "Harvard-FairVLMed10k": "/research/d5/gds/yzhong22/datasets/Harvard-FairVLMed10k",
}

In [43]:
df_index = pd.read_csv(os.path.join(exp_root, "exp_status.csv"))
df_index

,model,task,dataset,model_type,trainable_module,path,have_eval_result,have_prediction,have_gpt_score,model_family,if_finished,error
0,Qwen2-VL,vqa,SLAKE,general,NaN,vqa/SLAKE/Qwen2-VL/eval_seed0/Qwen2-VL-7B-Inst...,1,1,1,Qwen,1,NaN
1,Qwen25-VL,vqa,SLAKE,general,NaN,vqa/SLAKE/Qwen25-VL/eval_seed0/Qwen2.5-VL-7B-I...,1,1,1,Qwen,1,NaN
2,Gemma3,vqa,SLAKE,general,NaN,vqa/SLAKE/Gemma3/eval_seed0/gemma-3-4b-it,1,1,1,Gemma,1,NaN
3,MedGemma,vqa,SLAKE,medical,NaN,vqa/SLAKE/MedGemma/eval_seed0/medgemma-4b-it,1,1,1,Gemma,1,NaN
4,InternVL3,vqa,SLAKE,general,NaN,vqa/SLAKE/InternVL3/eval_seed0/InternVL3-8B-hf,1,1,1,Intern,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
215,gemini-2.5-pro,vqa,OmniMedVQA,medical,ucagent,vqa/OmniMedVQA/gemini-2.5-pro/eval_seed0/none/...,0,0,0,Gemini,0,NaN
216,o3,vqa,OmniMedVQA,medical,mdagent,vqa/OmniMedVQA/o3/eval_seed0/none/mdagent,0,0,0,o-family,0,NaN
217,o3,vqa,OmniMedVQA,medical,ucagent,vqa/OmniMedVQA/o3/eval_seed0/none/ucagent,0,0,0,o-family,0,NaN
218,Lingshu,vqa,OmniMedVQA,medical,mdagent,vqa/OmniMedVQA/Lingshu/eval_seed0/Lingshu-7B/m...,1,1,0,Qwen,1,NaN


In [44]:
import re
from collections import defaultdict
import math
from typing import Optional, Iterable


def extract_choice_letter(response: str, choices: Iterable[str] = tuple("ABCDE")) -> Optional[str]:
    """
    Extract a single-letter multi-choice answer (e.g., A/B/C/D/E) from a model response.
    Supports both uppercase and lowercase letters.
    """
    if response is None:
        return None

    text = response.strip()
    if not text:
        return None

    # normalize whitespace
    t = re.sub(r"\s+", " ", text)

    choices_set = set(c.upper() for c in choices)
    if not choices_set:
        return None

    # Helper: validate and normalize
    def ok(letter: str) -> Optional[str]:
        u = letter.upper()
        return u if u in choices_set else None

    # 1) Explicit cues: "answer: c", "final answer is (b)"
    cue_patterns = [
        r"(?:final\s+answer|answer\s+is|answer|ans|choice|option)\s*[:\-]?\s*[\(\[]?\s*([A-Za-z])\s*[\)\]]?",
        r"(?:correct\s+answer\s+is|the\s+correct\s+answer\s+is)\s*[\(\[]?\s*([A-Za-z])\s*[\)\]]?",
    ]
    for pat in cue_patterns:
        m = re.search(pat, t, flags=re.IGNORECASE)
        if m:
            letter = ok(m.group(1))
            if letter:
                return letter

    # 2) Parenthesized choices: "(b)"
    m = re.search(r"[\(\[]\s*([A-Za-z])\s*[\)\]]", t)
    if m:
        letter = ok(m.group(1))
        if letter:
            return letter

    # 3) Markdown bold: "**c**"
    m = re.search(r"\*\*\s*([A-Za-z])\s*\*\*", t)
    if m:
        letter = ok(m.group(1))
        if letter:
            return letter

    # 4) Fallback: standalone letter token
    m = re.search(r"\b([A-Za-z])\b", t)
    if m:
        letter = ok(m.group(1))
        if letter:
            return letter

    return None

In [45]:
import tqdm
import json
import numpy as np


datasets_expect_closed_test_num = {
    "SLAKE": 416,
    "PathVQA": 3362,
    "VQA-RAD": 251,
    "Harvard-FairVLMed10k": 1996,
    "MedXpertQA": 400,
    "OmniMedVQA": 6186,
}

bootstrap_indices = {}

seed = 42
rng = np.random.default_rng(seed)
n_bootstraps = 1000

skip_list = []

acc = []
ci_low = []
ci_high = []

for i in tqdm.tqdm(range(len(df_index))):
    row = df_index.iloc[i]

    if row["if_finished"] != 1:
        continue

    dataset = row["dataset"]
    path = row["path"]
    save_file = os.path.join(exp_root, path, "bootstrap_closed_meta.json")
    if any([x in path for x in skip_list]):
        continue

    # if os.path.exists(save_file):
    #     continue

    with open(os.path.join(exp_root, path, "predictions.json")) as fp:
        data = json.load(fp)

    accuracy_list = []

    for item in data:
        question_type = item["question_type"]

        if question_type == "open":
            continue
        else:
            answer = item["answer"].lower()
            pred = item["prediction"].lower()
            if dataset in ["MedXpertQA", "OmniMedVQA"]:
                question_type = "multi-choice"

            if question_type in ["yes/no", "closed"]:
                accuracy = 1 if answer in pred else 0
            elif question_type == "multi-choice":
                answer_letter = extract_choice_letter(pred, tuple("ABCDEF"))
                if answer_letter is None:
                    accuracy = 0
                else:
                    accuracy = int(str.lower(answer_letter) == answer)

            accuracy_list.append(accuracy)

    assert len(accuracy_list) == datasets_expect_closed_test_num[dataset]

    accuracy_list = np.array(accuracy_list)

    # generate boot indices for each dataset
    if dataset not in bootstrap_indices.keys():
        indices = []
        for _ in range(n_bootstraps):
            indices.append(
                rng.integers(0, datasets_expect_closed_test_num[dataset], datasets_expect_closed_test_num[dataset])
            )
        bootstrap_indices[dataset] = indices

    bootstrap_results = {}
    for _ in range(n_bootstraps):
        sample_idx = bootstrap_indices[dataset][_]

        bootstrap_results[f"boots_{_}"] = {
            "idx": sample_idx.astype(int).tolist(),
            "accuracy": accuracy_list[sample_idx].astype(int).tolist(),
        }

    with open(save_file, "w") as fp:
        json.dump(bootstrap_results, fp)

  0%|          | 0/220 [00:00<?, ?it/s]

100%|██████████| 220/220 [06:10<00:00,  1.68s/it]


In [35]:
row["if_finished"]

np.int64(1)

In [21]:
with open(
    "/media/yesindeed/DATADRIVE1/mount/remote_cse/experiments/med_vlm_benchmark/merged/vqa/SLAKE/Qwen2-VL/eval_seed0/Qwen2-VL-7B-Instruct/bootstrap_closed_meta.json"
) as fp:
    data = json.load(fp)

data["boots_0"]["accuracy"]

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
